# Synthetic Text Diversity Evaluation

This notebook evaluates lexical diversity of synthetic data using metrics from Texygen and related benchmarks:

1. **Distinct-n** – Fraction of unique n-grams in the corpus. Higher values indicate more diverse text; low values suggest mode collapse or repetition.
2. **Self-BLEU** – BLEU of each synthetic sentence against the rest of the corpus. Lower values indicate more diverse generations; high Self-BLEU warns of repetitive or low-entropy text.

## 1. Setup

In [1]:
!pip install -q pandas nltk

In [2]:
import pandas as pd
from typing import List, Tuple
import nltk

nltk.download("punkt_tab", quiet=True)

True

## 2. Diversity Metrics

In [3]:
def tokenize(text: str) -> List[str]:
    """Tokenize text (whitespace split). Swap for language-specific tokenizer if needed."""
    return str(text).strip().split() if pd.notna(text) and str(text).strip() else []


def get_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    """Extract n-grams from token list."""
    return [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)] if len(tokens) >= n else []


def compute_distinct_n(texts: List[str], n: int = 2) -> dict:
    """
    Compute Distinct-n: fraction of unique n-grams in the corpus.
    Higher = more diverse. Low values suggest mode collapse or repetition.

    distinct_n = |unique n-grams| / |total n-grams|
    """
    all_ngrams = []
    for text in texts:
        tokens = tokenize(text)
        all_ngrams.extend(get_ngrams(tokens, n))

    total = len(all_ngrams)
    unique = len(set(all_ngrams))
    distinct = unique / total if total > 0 else 0.0

    return {"unique_ngrams": unique, "total_ngrams": total, f"distinct_{n}": distinct}

In [4]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


def compute_self_bleu(texts: List[str], max_n: int = 4, sample_size: int = None) -> dict:
    """
    Compute Self-BLEU: BLEU of each sentence against the rest of the corpus.
    Lower = more diverse. High Self-BLEU warns of repetitive or low-entropy text.

    Uses geometric mean of BLEU-1 to BLEU-4 (Texygen-style).
    """
    tokenized = [tokenize(t) for t in texts if tokenize(t)]
    if len(tokenized) < 2:
        return {"self_bleu": 0.0, "n_sentences": len(tokenized)}

    if sample_size and len(tokenized) > sample_size:
        import random
        tokenized = random.sample(tokenized, sample_size)

    smoothing = SmoothingFunction().method1
    scores = []

    for i, hyp in enumerate(tokenized):
        refs = [tok for j, tok in enumerate(tokenized) if j != i]
        if not refs or not hyp:
            continue
        bleu_n = []
        for n in range(1, max_n + 1):
            if len(hyp) >= n:
                w = [1.0 / n] * n
                s = sentence_bleu(refs, hyp, weights=tuple(w), smoothing_function=smoothing)
                bleu_n.append(s)
        if bleu_n:
            geo_mean = 1.0
            for s in bleu_n:
                geo_mean *= s
            geo_mean **= 1.0 / len(bleu_n)
            scores.append(geo_mean)

    avg = sum(scores) / len(scores) if scores else 0.0
    return {"self_bleu": avg, "n_sentences": len(scores)}

## 3. Load Data

In [10]:
# Use fixed path to processed synthetic data
processed_data_path = r"F:\GitHub\topicmodeling\data\data_processed\train_1071_para_final.csv"

if not os.path.isfile(processed_data_path):
    raise FileNotFoundError(
        f"Could not find synthetic data file at '{processed_data_path}'.\n"
        "Please ensure the file exists and the path is correct."
    )

synthetic_df = pd.read_csv(processed_data_path)
text_column = "Sentence_clean" if "Sentence_clean" in synthetic_df.columns else "Sentence"
synthetic_texts = synthetic_df[text_column].dropna().astype(str).tolist()
print(f"Synthetic dataset: {len(synthetic_texts)} samples")

Synthetic dataset: 7957 samples


In [11]:
print(f"Current DATA_DIR being used: {DATA_DIR}")
print(f"synthetic_path: {synthetic_path}")
print(f"original_path: {original_path}")
print(f"Directory exists: {os.path.exists(DATA_DIR)}")
print("Files in DATA_DIR:", os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else "DATA_DIR does not exist")

Current DATA_DIR being used: data/processed
synthetic_path: data/processed\train_1071_para_final
original_path: data/processed\train_processed.csv
Directory exists: True
Files in DATA_DIR: ['.DS_Store', 'test_processed.csv', 'train_1071_para_final', 'train_processed.csv', 'val_processed.csv']


## 4. Compute Diversity Metrics

In [12]:
# Distinct-n (n=1,2,3)
print("Computing Distinct-n...")
distinct_1 = compute_distinct_n(synthetic_texts, n=1)
distinct_2 = compute_distinct_n(synthetic_texts, n=2)
distinct_3 = compute_distinct_n(synthetic_texts, n=3)

print("\n=== Distinct-n ===")
print(f"Distinct-1: {distinct_1['distinct_1']:.4f} (unique unigrams: {distinct_1['unique_ngrams']:,} / {distinct_1['total_ngrams']:,})")
print(f"Distinct-2: {distinct_2['distinct_2']:.4f} (unique bigrams: {distinct_2['unique_ngrams']:,} / {distinct_2['total_ngrams']:,})")
print(f"Distinct-3: {distinct_3['distinct_3']:.4f} (unique trigrams: {distinct_3['unique_ngrams']:,} / {distinct_3['total_ngrams']:,})")

Computing Distinct-n...

=== Distinct-n ===
Distinct-1: 0.0442 (unique unigrams: 5,014 / 113,416)
Distinct-2: 0.4755 (unique bigrams: 50,150 / 105,459)
Distinct-3: 0.8289 (unique trigrams: 80,827 / 97,506)


In [13]:
# Self-BLEU (use sample_size=500 for faster run on large corpora)
print("Computing Self-BLEU...")
self_bleu_result = compute_self_bleu(synthetic_texts, sample_size=500)
print(f"\n=== Self-BLEU ===")
print(f"Self-BLEU: {self_bleu_result['self_bleu']:.4f} (n_sentences: {self_bleu_result['n_sentences']})")
print("\nInterpretation: Lower Self-BLEU = more diverse. High Self-BLEU (>0.5) suggests repetitive text.")

Computing Self-BLEU...

=== Self-BLEU ===
Self-BLEU: 0.2989 (n_sentences: 500)

Interpretation: Lower Self-BLEU = more diverse. High Self-BLEU (>0.5) suggests repetitive text.


## 5. Optional: Compare with Original Data

In [14]:
if os.path.exists(original_path):
    original_df = pd.read_csv(original_path)
    orig_text_col = "Sentence_clean" if "Sentence_clean" in original_df.columns else "Sentence"
    original_texts = original_df[orig_text_col].dropna().astype(str).tolist()
    print(f"Original dataset: {len(original_texts)} samples")

    orig_d1 = compute_distinct_n(original_texts, n=1)
    orig_d2 = compute_distinct_n(original_texts, n=2)
    orig_sb = compute_self_bleu(original_texts, sample_size=500)

    summary = pd.DataFrame({
        "Metric": ["Distinct-1", "Distinct-2", "Distinct-3", "Self-BLEU"],
        "Original": [orig_d1["distinct_1"], orig_d2["distinct_2"], compute_distinct_n(original_texts, 3)["distinct_3"], orig_sb["self_bleu"]],
        "Synthetic": [distinct_1["distinct_1"], distinct_2["distinct_2"], distinct_3["distinct_3"], self_bleu_result["self_bleu"]],
    })
    summary["Synthetic - Original"] = summary["Synthetic"] - summary["Original"]
    display(summary)
else:
    print("Original file not found. Skipping comparison.")

Original dataset: 5548 samples


,Metric,Original,Synthetic,Synthetic - Original
0,Distinct-1,0.059729,0.044209,-0.015520
1,Distinct-2,0.593515,0.475540,-0.117975
2,Distinct-3,0.930389,0.828944,-0.101445
3,Self-BLEU,0.258007,0.298858,0.040851
